# RQ2 — Why does Uniform/Sandwich win? (read-only T4 x2)

Mechanism audit of temporal SW predictiveness, pair diversity/exposure, momentum interference, and stage dependence. It consumes the completed four-way U/R/SW/RG run and never trains or updates a model.

## Required inputs

1. Exactly one recovered/completed four-way root containing `e2e_pairwise_pilot_v2`.
2. CIFAR-100 containing `cifar-100-python/{train,test,meta}`.
3. Gate-A output containing `gate_a_summary.json`.
4. Kaggle secret `github_token`. Enable T4 x2.

Do not attach multiple old copies of the four-way output.

In [ ]:
import os, subprocess, sys, json, time, zipfile, importlib, hashlib
from pathlib import Path
from IPython.display import display
from kaggle_secrets import UserSecretsClient
github_token = UserSecretsClient().get_secret('github_token')
assert github_token, 'Missing Kaggle secret github_token'
PROJECT_ROOT = Path('/kaggle/working/new-pruning')
askpass = Path('/kaggle/working/.github_git_askpass.py')
askpass.write_text("#!/usr/bin/env python3\nimport os,sys\np=sys.argv[1] if len(sys.argv)>1 else ''\nprint('x-access-token' if 'Username' in p else os.environ['GITHUB_TOKEN_RUNTIME'])\n")
askpass.chmod(0o700)
env = os.environ.copy(); env.update({'GIT_ASKPASS':str(askpass),'GIT_TERMINAL_PROMPT':'0','GITHUB_TOKEN_RUNTIME':github_token})
try:
    command = ['git','-C',str(PROJECT_ROOT),'pull','--ff-only'] if (PROJECT_ROOT/'.git').is_dir() else ['git','clone','https://github.com/duyh80456-code/new-pruning.git',str(PROJECT_ROOT)]
    subprocess.run(command, env=env, check=True)
finally:
    askpass.unlink(missing_ok=True); github_token = None
source_candidates = [PROJECT_ROOT] + [path.parent for path in Path('/kaggle/input').rglob('rq2_uniform_win_diagnostics.py')]
source_candidates = [path.resolve() for path in source_candidates if (path/'rq2_uniform_win_diagnostics.py').is_file()]
assert source_candidates, 'Diagnostic source not found; sync/push the new source files first.'
SOURCE_CODE_ROOT = PROJECT_ROOT.resolve() if (PROJECT_ROOT/'rq2_uniform_win_diagnostics.py').is_file() else source_candidates[0]
os.chdir(SOURCE_CODE_ROOT); sys.path.insert(0, str(SOURCE_CODE_ROOT))
import torch
assert torch.cuda.device_count() == 2, f'Select T4 x2; detected {torch.cuda.device_count()}'
GPU_IDS = (0,1)
GIT_COMMIT = subprocess.run(['git','-C',str(SOURCE_CODE_ROOT),'rev-parse','HEAD'],capture_output=True,text=True).stdout.strip()
print('Code source:', SOURCE_CODE_ROOT); print('Commit:', GIT_COMMIT)

## Resolve and audit the completed run

In [ ]:
import rq2_e2e_pairwise_pilot as pilot
import rq2_uniform_win_diagnostics as diagnostic
import scripts.run_uniform_win_diagnostics as runner
pilot = importlib.reload(pilot); diagnostic = importlib.reload(diagnostic); runner = importlib.reload(runner)
INPUT_ROOT = Path('/kaggle/input')
DATASET_ROOT = pilot.find_cifar100_root(INPUT_ROOT)
GATE_A_SUMMARY = pilot.find_gate_a_summary(INPUT_ROOT)
ROOT = pilot.materialize_progress(INPUT_ROOT, '/kaggle/working/e2e_pairwise_pilot_v2', '/kaggle/working/materialized-uniform-win-input')
required = [ROOT/'frozen_protocol.json', ROOT/'resolved_config.yaml', ROOT/'common_warmup/epoch_010.pt', ROOT/'rpgeo_frozen_retention.json']
for method in diagnostic.METHODS:
    required += [ROOT/method/'checkpoints/epoch_050.pt', ROOT/method/'checkpoints/epoch_100.pt', ROOT/method/'metrics.csv', ROOT/method/'pair_stats.csv', ROOT/method/'dense_metrics.csv']
    required += [ROOT/method/'sw_policies'/f'epoch_{epoch:03d}.npz' for epoch in range(10,100,10)]
missing = [str(path) for path in required if not path.is_file()]
assert not missing, f'Incomplete recovered four-way input: {missing[:20]}'
OUTPUT_DIR = Path('/kaggle/working/rq2-uniform-win-diagnostics')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
input_protocol = {'status':'READ_ONLY_MECHANISM_AUDIT','training_authorized':False,'optimizer_steps_authorized':0,'methods':list(diagnostic.METHODS),'checkpoint_epochs':[10,50,100],'epoch_50_optimizer_reset_handling':'pre-reset and protocol-correct zero-momentum post-reset both reported','git_commit':GIT_COMMIT,'common_epoch10_sha256':pilot._sha256(ROOT/'common_warmup/epoch_010.pt'),'rpgeo_epoch100_sha256':pilot._sha256(ROOT/'resource_geo/checkpoints/epoch_100.pt')}
(OUTPUT_DIR/'diagnostic_protocol.json').write_text(json.dumps(input_protocol,indent=2)+'\n')
print('Experiment root:',ROOT); print('CIFAR-100:',DATASET_ROOT); print('Gate A:',GATE_A_SUMMARY)

## CPU audit: temporal prediction, diversity, exposure, stages

In [ ]:
started = time.perf_counter()
temporal_result = diagnostic.temporal_predictive_audit(ROOT, OUTPUT_DIR)
diversity_result = diagnostic.diversity_and_stage_audit(ROOT, OUTPUT_DIR)
print(temporal_result); print(diversity_result)
display(__import__('pandas').read_csv(OUTPUT_DIR/'temporal_sw_forecast.csv').groupby(['method','lag'],as_index=False)[['pearson','spearman','top_quartile_overlap']].mean())
display(__import__('pandas').read_csv(OUTPUT_DIR/'sw_to_future_gradient.csv'))
display(__import__('pandas').read_csv(OUTPUT_DIR/'pair_diversity_by_epoch.csv').groupby('method',as_index=False)[['expected_entropy','expected_effective_support','realized_effective_support']].mean())
display(__import__('pandas').read_csv(OUTPUT_DIR/'stage_summary.csv'))
print(f'CPU audit: {(time.perf_counter()-started)/60:.1f} min')

## GPU read-only momentum probe

At E50 the notebook reports both the saved pre-reset momentum and the zero momentum actually used when the protocol creates a new optimizer for E51.

In [ ]:
started = time.perf_counter()
runtime = runner.run_momentum_states(ROOT, DATASET_ROOT, GATE_A_SUMMARY, OUTPUT_DIR, gpu_ids=GPU_IDS)
summary = diagnostic.finalize_uniform_win_diagnostic(ROOT, OUTPUT_DIR)
display(runtime); print(json.dumps(summary,indent=2))
display(__import__('pandas').read_csv(OUTPUT_DIR/'momentum_policy_summary.csv'))
print(f'Momentum audit: {(time.perf_counter()-started)/60:.1f} min')

## Export diagnostic-only artifacts

In [ ]:
required_outputs = ['diagnostic_protocol.json','temporal_sw_forecast.csv','sw_to_future_gradient.csv','sw_to_future_width_accuracy.csv','sw_to_future_training_loss.csv','pair_diversity_by_epoch.csv','pair_exposure_vs_future_gradient.csv','stage_summary.csv','momentum_policy_summary.csv','momentum_width_summary.csv','momentum_pair_summary.csv','uniform_win_diagnostic_summary.json']
missing = [name for name in required_outputs if not (OUTPUT_DIR/name).is_file()]
assert not missing, f'Missing diagnostic outputs: {missing}'
bundle = Path('/kaggle/working/rq2-uniform-win-mechanism-diagnostics.zip')
with zipfile.ZipFile(bundle,'w',compression=zipfile.ZIP_DEFLATED,allowZip64=True) as archive:
    for path in OUTPUT_DIR.rglob('*'):
        if path.is_file(): archive.write(path,Path('rq2-uniform-win-diagnostics')/path.relative_to(OUTPUT_DIR))
print('Persist:',bundle,f'{bundle.stat().st_size/2**20:.1f} MiB'); bundle